# Algorithmic Systems Design: Smart Spell-Checker & Autocorrect Pipeline
**Authors:** Jakub Habib, Ulugbek Tojiboev

### System Objective
A real-time pipeline that verifies user text input and instantly suggests the top 3 statistically probable corrections for typos. The system chains three distinct algorithms to process data efficiently.

In [3]:
import re

class Vocabulary:
    def __init__(self):
        # Maps word (str) to its frequency (int)
        self.word_frequencies = {}

    def clean_text(self, text):
        """Normalize data: remove non-alphabetic characters and convert to lowercase."""
        if not text:
            return ""
        cleaned = re.sub(r'[^a-zA-Z]', '', text)
        return cleaned.lower()

    def load_dictionary(self, raw_data):
        """Build the Hash Map from a list of (word, frequency) tuples."""
        for word, freq in raw_data:
            cleaned_word = self.clean_text(word)
            if cleaned_word:
                self.word_frequencies[cleaned_word] = freq
        print(f"Loaded {len(self.word_frequencies)} words into the dictionary.")

    def is_valid_word(self, word):
        """Fast O(1) validation. Checks if the word exists in the dictionary."""
        cleaned_word = self.clean_text(word)
        if not cleaned_word:
            return True # Ignore empty strings

        # O(1) Hash Map lookup
        return cleaned_word in self.word_frequencies

# --- MOCK TEST FOR STAGE 1 ---
if __name__ == "__main__":
    mock_database = [
        ("the", 100000),
        ("tea", 500),
        ("ten", 3000),
        ("hello", 50000),
        ("definitely", 20000)
    ]

    vocab = Vocabulary()
    vocab.load_dictionary(mock_database)

    test_word_1 = "Hello!"      
    test_word_2 = "definetly"   

    print(f"Is 'Hello!' valid? {vocab.is_valid_word(test_word_1)}")
    print(f"Is 'definetly' valid? {vocab.is_valid_word(test_word_2)}")

Loaded 5 words into the dictionary.
Is 'Hello!' valid? True
Is 'definetly' valid? False


In [2]:
import heapq

class TopKRanker:
    def __init__(self, k=3):
        self.k = k

    def get_top_k(self, candidates, word_frequencies):
        min_heap = []
        
        for word in candidates:
            # Default to 0 if word is somehow missing from vocabulary
            freq = word_frequencies.get(word, 0) 
            
            # Push tuple (frequency, word) into the min-heap
            heapq.heappush(min_heap, (freq, word))
            
            # Maintain strict heap size of k to achieve O(C log k) time
            if len(min_heap) > self.k:
                heapq.heappop(min_heap)
        
        # Sort the remaining k elements in descending order (O(k log k))
        result = sorted(min_heap, key=lambda x: x[0], reverse=True)
        
        # Extract just the words
        return [word for freq, word in result]

# --- MOCK TEST FOR STAGE 3 ---
if __name__ == "__main__":
    # Mock data representing the Hash Map from Stage 1
    mock_frequencies = {
        "tea": 500,
        "ten": 3000,
        "the": 100000,
        "ted": 150,
        "tech": 15000
    }
    
    # Mock candidates representing the output from Stage 2 (Levenshtein)
    # Simulated typo: "teh"
    mock_candidates = ["tea", "ten", "the", "ted", "tech"]
    
    ranker = TopKRanker(k=3)
    top_suggestions = ranker.get_top_k(mock_candidates, mock_frequencies)
    
    print(f"Raw Candidates: {mock_candidates}")
    print(f"Top 3 Suggestions: {top_suggestions}")

Raw Candidates: ['tea', 'ten', 'the', 'ted', 'tech']
Top 3 Suggestions: ['the', 'tech', 'ten']


# Here is what the code actually does and how to present it:

I finished the full Python script. Basically, it’s a 3-stage pipeline that takes a user's word and checks/fixes it. Each stage connects directly to the next one.

---

## 1. How the Data Flows (The Pipeline)

* **Step 1:** The user types a word (like `"teh"`).
* **Step 2:** Stage 1 checks the **Hash Set**. Since `"teh"` isn't there, it flags it as a typo and passes the string `"teh"` to Stage 2.
* **Step 3:** Stage 2 filters out huge words using our **length filter**, then runs the **Levenshtein matrix math** on close words. It finds raw candidates like `["the", "then", "tea", "ten"]` and passes this list to Stage 3.
* **Step 4:** Stage 3 takes that list, looks up how popular each word is, and throws them into a **Min-Heap**. It keeps only the top 3 most popular words and gives them to the user.

---

## 2. Fast Summary Table (For our Slides)

Here is the exact breakdown we can use for our technical deep dive slide:

| Stage | What it does | Data Structure Used | Time Complexity (Big O) | Space Complexity |
| :--- | :--- | :--- | :--- | :--- |
| **1. Validation** | Instantly checks if the word is spelled right. | **Hash Set** | $O(1)$ (Super fast lookup) | $O(V)$ (Stores dictionary) |
| **2. Correction** | Finds words that look similar to the typo. | **2D DP Matrix** | $O(N \times M)$ (Our bottleneck!) | $O(N \times M)$ (Matrix size) |
| **3. Ranking** | Grabs only the top 3 most popular corrections. | **Min-Heap (Size 3)** | $O(C \log k) \rightarrow O(C)$ | $O(1)$ (Always size 3) |

*V = total words in dictionary, N & M = word lengths, C = number of candidates from Stage 2, k = 3.*

---

## 3. Two Things We MUST Defend in the Presentation

Our professor is definitely going to ask us *why* we chose these structures. Here are our exact answers:

### Defense 1: Why a Hash Set over a Binary Search Tree (BST) for Stage 1?
A BST makes us do $O(\log V)$ steps to find a word because it searches through branches. A Hash Set uses a hash function to jump straight to the word in $O(1)$ constant time. For a spell-checker gatekeeper, speed is everything, so Hash Set wins.

### Defense 2: Why a Min-Heap over Sorting for Stage 3?
If Stage 2 gives us 20 candidate words, sorting the whole list takes $O(C \log C)$ time. But we *only* need the top 3. By using a Min-Heap capped at size 3, we just slide words through it. If a low-frequency word drops to the bottom, we instantly pop it out. It keeps our memory locked at a constant size 3.

### Our Main Bottleneck: Stage 2
We need to explicitly tell the professor that **Stage 2 is our system bottleneck**. Running those nested loops to build a 2D matrix for Levenshtein distance ($O(N \times M)$) takes up almost all the CPU power. That's why we added our optimization line (`if abs(len(dict_word) - typo_len) <= 2`) to instantly skip giant words like `"refrigerator"` when the user types a short typo like `"teh"`.

In [1]:
import heapq

class SpellCheckerPipeline:
    def __init__(self, vocabulary_data):
        self.dictionary_set = set(vocabulary_data.keys())
        self.frequencies = vocabulary_data

    def _levenshtein_distance(self, s1, s2):
        m, n = len(s1), len(s2)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        
        for i in range(m + 1):
            dp[i][0] = i
        for j in range(n + 1):
            dp[0][j] = j
            
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if s1[i-1] == s2[j-1]:
                    dp[i][j] = dp[i-1][j-1]
                else:
                    dp[i][j] = 1 + min(
                        dp[i-1][j],    
                        dp[i][j-1],    
                        dp[i-1][j-1]   
                    )
        return dp[m][n]

    def process_word(self, input_word):
        normalized_word = input_word.lower().strip()
        
        # Stage 1: Hash Set Lookup
        if normalized_word in self.dictionary_set:
            return [normalized_word]
            
        # Stage 2: Candidate Generation with Length Filter
        candidates = []
        typo_len = len(normalized_word)
        
        for dict_word in self.dictionary_set:
            if abs(len(dict_word) - typo_len) <= 2:
                dist = self._levenshtein_distance(normalized_word, dict_word)
                if dist <= 2:
                    candidates.append(dict_word)
                    
        # Stage 3: Min-Heap Ranking
        min_heap = []
        k = 3
        
        for word in candidates:
            freq = self.frequencies.get(word, 0)
            heapq.heappush(min_heap, (freq, word))
            if len(min_heap) > k:
                heapq.heappop(min_heap)
                
        return [word for freq, word in sorted(min_heap, reverse=True)]


if __name__ == "__main__":
    mock_vocabulary = {
        "the": 5000,
        "then": 1200,
        "tea": 450,
        "ten": 300,
        "there": 2500,
        "apple": 800,
        "banana": 600,
        "to": 4000
    }
    
    pipeline = SpellCheckerPipeline(mock_vocabulary)
    
    # Test cases
    print("Result for 'apple':", pipeline.process_word("apple"))
    print("Result for 'teh':", pipeline.process_word("teh"))

Result for 'apple': ['apple']
Result for 'teh': ['the', 'to', 'then']
